# 2. Chat App with Feedback

Let's build a chat app that:
- Lets the user select a course and ask a question
- Runs the RAG pipeline and shows the answer
- Has thumbs up / thumbs down buttons for feedback

## Setting up the RAG pipeline

We use `RAGBase`, `MinsearchIndex`, `FaqHttpLoader`, and `OllamaClient` from `src/`.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.insert(0, '../../')

import dotenv
dotenv.load_dotenv('../../.env', override=True)

True

In [2]:
from src import FaqHttpLoader, MinsearchIndex, RAGBase, OllamaClient

OLLAMA_MODEL = 'granite4.1:3b'

INSTRUCTIONS = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

# Load documents and build index
loader = FaqHttpLoader()
documents = loader.load()
print(f'Loaded {len(documents)} documents')

index = MinsearchIndex(documents)
llm_client = OllamaClient(num_ctx=16384)

assistant = RAGBase(
    index=index,
    llm_client=llm_client,
    llm_model=OLLAMA_MODEL,
    instructions=INSTRUCTIONS,
)

Loaded 1208 documents


## LLM call with metrics

For monitoring we need response time and token usage.

`OllamaClient.complete_with_metrics` calls `/api/chat` directly and captures
Ollama's native `prompt_eval_count` / `eval_count` fields alongside wall-clock time.

In [3]:
# Quick smoke-test of complete_with_metrics
test_prompt = "What is Docker?"
test_instructions = "Answer briefly."

answer, tokens, response_time = llm_client.complete_with_metrics(
    test_prompt, test_instructions, OLLAMA_MODEL
)

print(f'Answer: {answer[:100]}...')
print(f'Response time: {response_time:.2f}s')
print(f'Tokens: {tokens}')

Answer: Docker is an open-source platform that automates the deployment, scaling, and management of applicat...
Response time: 8.55s
Tokens: {'prompt_tokens': 20, 'completion_tokens': 45, 'total_tokens': 65}


## Relevance evaluation (LLM-as-a-judge)

We use the same structured-output pattern from notebook 04:
`ollama_client.beta.chat.completions.parse` with a Pydantic model.

This avoids fragile JSON parsing — the result is always well-formed.

In [4]:
from src.monitoring import evaluate_relevance

relevance, explanation = evaluate_relevance(
    question='How do I install Docker on Ubuntu?',
    answer='You can install Docker by running: sudo apt-get install docker.io',
    model=OLLAMA_MODEL,
)

print(f'Relevance: {relevance}')
print(f'Explanation: {explanation}')

Relevance: RELEVANT
Explanation: The generated answer directly instructs how to install Docker on Ubuntu using the command 'sudo apt-get install docker.io', which is the standard method for installing Docker Community Edition (CE) on Ubuntu systems. This directly addresses the question.


In [5]:
# Test with a non-relevant answer
relevance2, explanation2 = evaluate_relevance(
    question='How do I install Docker on Ubuntu?',
    answer='The capital of France is Paris.',
    model=OLLAMA_MODEL,
)

print(f'Relevance: {relevance2}')
print(f'Explanation: {explanation2}')

Relevance: NON_RELEVANT
Explanation: The generated answer discusses the capital of France, which has no connection to installing Docker on Ubuntu. It does not address or relate to the question posed.


## Cost calculation

Ollama runs locally for free, but we **simulate cost** using GPT-4o-mini pricing
so the monitoring dashboard shows realistic numbers and the metric stays
meaningful if the backend is ever swapped to OpenAI.

Rates used (per 1M tokens):
- Input: **$0.15**
- Output: **$0.60**

In [6]:
from src.monitoring import calculate_cost

cost = calculate_cost(OLLAMA_MODEL, tokens)
print(f'Simulated cost: ${cost:.6f}')
print(f'  prompt_tokens={tokens["prompt_tokens"]} × $0.15/1M = ${tokens["prompt_tokens"] * 0.15 / 1_000_000:.6f}')
print(f'  completion_tokens={tokens["completion_tokens"]} × $0.60/1M = ${tokens["completion_tokens"] * 0.60 / 1_000_000:.6f}')

Cost: $0.000030


## `get_answer` — full pipeline with metrics

Ties everything together:
1. `rag.search` → retrieve relevant FAQ docs
2. `rag.build_context` + `rag.build_prompt` → format prompt
3. `llm_client.complete_with_metrics` → answer + tokens + time
4. `evaluate_relevance` → LLM-as-a-judge score
5. `calculate_cost` → cost estimate

In [7]:
from src.monitoring import get_answer

result = get_answer(
    rag=assistant,
    llm_client=llm_client,
    question='How do I run Docker on Windows?',
    course='data-engineering-zoomcamp',
    model=OLLAMA_MODEL,
)

for key, value in result.items():
    if key in ('answer', 'relevance_explanation'):
        print(f'{key}: {str(value)[:120]}')
    else:
        print(f'{key}: {value}')

answer: To run Docker on Windows, first ensure that Hyper-V is enabled if you are using a Pro or Enterprise edition of Windows 1
response_time: 45.3304877281189
relevance: RELEVANT
relevance_explanation: The answer directly addresses how to run Docker on Windows by providing key steps such as enabling Hyper-V for Pro/Enter
model_used: granite4.1:3b
prompt_tokens: 1128
completion_tokens: 92
total_tokens: 1220
ollama_cost: 0.00022439999999999998


## Building the Streamlit app

The full app lives in `app.py` at the project root. Run it with:

```bash
uv run streamlit run app.py
```

Key design decisions vs. the lesson's reference code:

| Lesson reference | Our implementation |
|---|---|
| `OpenAI()` | `OllamaClient(num_ctx=4096)` |
| `llm_with_metrics` standalone fn | `OllamaClient.complete_with_metrics` method |
| JSON parsing in `evaluate_relevance` | Pydantic structured output via `beta.chat.completions.parse` |
| PostgreSQL storage | In-memory `st.session_state` (next lesson adds DB) |
| `@st.cache_resource` | ✅ same — index built once per server process |

The `save_conversation` and `save_feedback` functions currently write to
`st.session_state`. PostgreSQL persistence will be added in lesson 03.

In [8]:
# Preview the app.py source
from pathlib import Path
print(Path('../../app.py').read_text())

"""
Streamlit chat app with feedback for the RAG pipeline.

Run with:
    uv run streamlit run app.py

Features:
- Course selector
- Question input → RAG answer via local Ollama
- Response time, relevance score, token usage display
- Thumbs up / thumbs down feedback buttons
- Conversation history stored in session state
"""

import uuid
from datetime import datetime

import streamlit as st

from src import FaqHttpLoader, MinsearchIndex, OllamaClient, RAGBase
from src.monitoring import get_answer

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------

OLLAMA_MODEL = "granite4.1:3b"

COURSES = [
    "data-engineering-zoomcamp",
    "machine-learning-zoomcamp",
    "mlops-zoomcamp",
    "llm-zoomcamp",
]

INSTRUCTIONS = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering 

## What each interaction saves

Each call to `get_answer` returns (and `save_conversation` stores):

| Field | Description |
|---|---|
| `answer` | LLM-generated answer |
| `response_time` | Wall-clock seconds for the LLM call |
| `relevance` | `RELEVANT` / `PARTLY_RELEVANT` / `NON_RELEVANT` |
| `relevance_explanation` | Judge's reasoning |
| `model_used` | Ollama model identifier |
| `prompt_tokens` | Tokens in the prompt |
| `completion_tokens` | Tokens in the response |
| `total_tokens` | Sum of above |
| `ollama_cost` | Simulated cost in USD (GPT-4o-mini rates: $0.15/1M input, $0.60/1M output) |

Feedback (`+1` / `-1`) is stored separately and linked by `conversation_id`.